# Summer Analytics 2026 - Week 2 Hackathon
## E-Commerce Conversion Prediction Challenge
**Approach:** Ensemble of XGBoost + LightGBM + RandomForest with feature engineering and threshold tuning.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import f1_score, classification_report
import xgboost as xgb
import lightgbm as lgb

print('Libraries imported successfully')

## Step 1: Load Data

In [ ]:
train = pd.read_csv('train.csv')
public_test = pd.read_csv('public_test.csv')
private_test = pd.read_csv('private_test.csv')

print('Train:', train.shape)
print('Public Test:', public_test.shape)
print('Private Test:', private_test.shape)

# public_test has labels - use it for training too
full_train = pd.concat([train, public_test], axis=0, ignore_index=True)
print('\nCombined training set:', full_train.shape)
print('Target distribution:\n', full_train['Converted'].value_counts())

## Step 2: Exploratory Data Analysis

In [ ]:
print('Missing values in training set:')
print(full_train.isnull().sum())

print('\nFeature correlations with Converted:')
numeric_cols = full_train.select_dtypes(include='number').columns.drop('User_ID')
print(full_train[numeric_cols].corr()['Converted'].sort_values(ascending=False))

print('\nDevice_Type values:', full_train['Device_Type'].unique())
print('Traffic_Source values:', full_train['Traffic_Source'].unique())

## Step 3: Feature Engineering

Key insight from EDA: `Pages_Viewed` and `Products_Viewed` have the strongest correlation with conversion (~0.31). Feature engineering focuses on:
- Polynomial interaction terms between engagement features
- Discount amplification signals
- User loyalty indicators
- Traffic source and device flags

In [ ]:
# Fit label encoders on all data combined
all_data = pd.concat([full_train.drop('Converted', axis=1), private_test], axis=0, ignore_index=True)
le_device = LabelEncoder()
le_traffic = LabelEncoder()
le_device.fit(all_data['Device_Type'].fillna('Unknown'))
le_traffic.fit(all_data['Traffic_Source'].fillna('Unknown'))

def engineer_features(df):
    df = df.copy()
    
    # Encode categoricals
    df['Device_enc'] = le_device.transform(df['Device_Type'].fillna('Unknown'))
    df['Traffic_enc'] = le_traffic.transform(df['Traffic_Source'].fillna('Unknown'))
    
    # Engagement interactions (strongest signal features)
    df['Pages_x_Products'] = df['Pages_Viewed'] * df['Products_Viewed']
    df['Pages_sq'] = df['Pages_Viewed'] ** 2
    df['Products_sq'] = df['Products_Viewed'] ** 2
    df['Total_Engagement'] = df['Pages_Viewed'] + df['Products_Viewed'] * 2
    
    # Discount amplification
    df['Discount_x_Pages'] = df['Discount_Seen'] * df['Pages_Viewed']
    df['Discount_x_Products'] = df['Discount_Seen'] * df['Products_Viewed']
    df['Discount_x_Prev'] = df['Discount_Seen'] * df['Previous_Purchases']
    
    # Time-based depth metrics
    df['Time_per_Page'] = df['Time_On_Site'] / (df['Pages_Viewed'] + 0.001)
    df['Time_per_Product'] = df['Time_On_Site'] / (df['Products_Viewed'] + 0.001)
    
    # Purchase history
    df['Loyal_User'] = (df['Previous_Purchases'] > 2).astype(int)
    df['New_User'] = (df['Previous_Purchases'] == 0).astype(int)
    
    # Device and channel flags
    df['Is_Mobile'] = (df['Device_Type'] == 'Mobile').astype(int)
    df['Is_Desktop'] = (df['Device_Type'] == 'Desktop').astype(int)
    df['Is_Paid'] = (df['Traffic_Source'] == 'Paid Ads').astype(int)
    df['Is_Email'] = (df['Traffic_Source'] == 'Email').astype(int)
    df['Is_Social'] = (df['Traffic_Source'] == 'Social Media').astype(int)
    
    # High engagement thresholds
    df['High_Pages'] = (df['Pages_Viewed'] >= 10).astype(int)
    df['High_Products'] = (df['Products_Viewed'] >= 5).astype(int)
    
    # Geographic income interaction
    df['Tier1_High_Income'] = ((df['City_Tier'] == 1) & (df['Income'] > 60000)).astype(int)
    
    return df

full_fe = engineer_features(full_train)
priv_fe = engineer_features(private_test)
print('Feature engineering complete')

## Step 4: Prepare Feature Matrix

In [ ]:
FEATURE_COLS = [
    'Age', 'Income', 'City_Tier', 'Device_enc', 'Traffic_enc',
    'Pages_Viewed', 'Products_Viewed', 'Time_On_Site', 'Previous_Purchases',
    'Discount_Seen', 'Browser_Version', 'Campaign_Code',
    'Pages_x_Products', 'Pages_sq', 'Products_sq', 'Total_Engagement',
    'Discount_x_Pages', 'Discount_x_Products', 'Discount_x_Prev',
    'Time_per_Page', 'Time_per_Product',
    'Loyal_User', 'New_User',
    'Is_Mobile', 'Is_Desktop', 'Is_Paid', 'Is_Email', 'Is_Social',
    'High_Pages', 'High_Products', 'Tier1_High_Income'
]

X = full_fe[FEATURE_COLS]
y = full_fe['Converted']
X_priv = priv_fe[FEATURE_COLS]

# Impute with median (robust to outliers)
imputer = SimpleImputer(strategy='median')
X_imp = imputer.fit_transform(X)
X_priv_imp = imputer.transform(X_priv)

print('Feature matrix:', X_imp.shape)
print('Private test matrix:', X_priv_imp.shape)

## Step 5: Model Definition & Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pos_weight = (y == 0).sum() / (y == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=600, max_depth=6, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
    scale_pos_weight=pos_weight,
    random_state=42, verbosity=0, eval_metric='logloss'
)

lgb_model = lgb.LGBMClassifier(
    n_estimators=600, max_depth=6, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.7, min_child_samples=20,
    is_unbalance=True, random_state=42, verbose=-1
)

rf_model = RandomForestClassifier(
    n_estimators=400, max_depth=10, min_samples_leaf=5,
    class_weight='balanced', random_state=42, n_jobs=-1
)

print('5-Fold Stratified Cross-Validation F1 Scores:')
for name, model in [('XGBoost', xgb_model), ('LightGBM', lgb_model), ('RandomForest', rf_model)]:
    scores = cross_val_score(model, X_imp, y, cv=cv, scoring='f1', n_jobs=-1)
    print(f'  {name}: {scores.mean():.4f} +/- {scores.std():.4f}')

## Step 6: Threshold Tuning via OOF Predictions

In [ ]:
# Generate out-of-fold probability estimates
xgb_oof = cross_val_predict(xgb_model, X_imp, y, cv=cv, method='predict_proba')[:, 1]
lgb_oof = cross_val_predict(lgb_model, X_imp, y, cv=cv, method='predict_proba')[:, 1]
rf_oof = cross_val_predict(rf_model, X_imp, y, cv=cv, method='predict_proba')[:, 1]

# Ensemble OOF with equal-ish weights
oof_ensemble = 0.35 * xgb_oof + 0.35 * lgb_oof + 0.30 * rf_oof

# Find optimal threshold for F1
best_threshold, best_f1 = 0.5, 0.0
for t in np.arange(0.30, 0.70, 0.01):
    preds = (oof_ensemble >= t).astype(int)
    f = f1_score(y, preds)
    if f > best_f1:
        best_f1 = f
        best_threshold = t

print(f'Optimal threshold: {best_threshold:.2f}')
print(f'Best OOF F1: {best_f1:.4f}')

final_oof_preds = (oof_ensemble >= best_threshold).astype(int)
print('\nClassification Report on OOF predictions:')
print(classification_report(y, final_oof_preds))

## Step 7: Train Final Models & Generate Predictions

In [ ]:
# Train on full dataset
xgb_model.fit(X_imp, y)
lgb_model.fit(X_imp, y)
rf_model.fit(X_imp, y)

# Ensemble probabilities on private test
xgb_proba = xgb_model.predict_proba(X_priv_imp)[:, 1]
lgb_proba = lgb_model.predict_proba(X_priv_imp)[:, 1]
rf_proba = rf_model.predict_proba(X_priv_imp)[:, 1]

ensemble_proba = 0.35 * xgb_proba + 0.35 * lgb_proba + 0.30 * rf_proba
final_predictions = (ensemble_proba >= best_threshold).astype(int)

print('Prediction distribution:', pd.Series(final_predictions).value_counts().to_dict())

## Step 8: Generate Submission File

In [ ]:
submission = pd.DataFrame({
    'User_ID': private_test['User_ID'],
    'Converted': final_predictions
})

submission.to_csv('submission.csv', index=False)
print('submission.csv created!')
submission.head(10)